# Simulating Fraud Transaction Datasets for Impossible Geographic Jumps

In this notebook, we use [DataFramer](https://www.dataframer.ai) to generate a large, labelled synthetic dataset of fraudulent transactions exhibiting **impossible geographic jumps**, giving fraud classifiers the rare-pattern coverage that real data cannot provide.

Real fraud data is scarce, class-imbalanced, and tightly governed. This notebook shows how to use [DataFramer](https://www.dataframer.ai) to generate a reality-grounded synthetic dataset where the primary fraud signal is transactions appearing in physically impossible locations in rapid succession.

### The Fraud Pattern: Impossible Geographic Jumps

When a stolen card number (not the physical card) is used, transactions appear in physically impossible locations in quick succession — New York at 9 AM, Los Angeles at 9:58 AM, Chicago at 11:22 AM, for example. No human can travel that fast, making geo-jump one of the strongest real-time fraud signals.

Key signals in the data:

- City sequences that are physically impossible within the elapsed time between transactions
- Short inter-transaction gaps (minutes to low hours) despite spanning thousands of miles
- `geo_jump_flag` marks each transaction as `geo_jump` or `normal`
- Legitimate transactions cluster geographically — the same city or nearby cities within a plausible travel window

### Why Synthetic Data?

- **Class imbalance**: Fraud rates in the wild are often < 0.5%. Models trained on real data struggle to learn rare patterns without oversampling.
- **Data access**: Real transaction data carries PII and is heavily access-controlled. Synthetic data sidesteps compliance barriers.
- **Pattern control**: Synthetic data lets you explicitly control the ratio of geo-jump vs normal transactions, the severity of the jumps, and edge cases your model needs to handle.
- **Labelling cost**: Labelling real fraud at scale requires expert fraud analysts. DataFramer generates labelled data from a small seed at a fraction of the cost.

### What We Build

Starting from a 10-row seed file that encodes impossible city-hop sequences, we use DataFramer to generate 100 synthetic transaction records with a controlled geo-jump distribution. The result is a labelled dataset ready to train or evaluate a fraud classifier focused on geographic anomaly detection.

## Setup

In [ ]:
import os

!git clone https://github.com/aimonlabs/dataframer-docs-public.git
os.chdir("dataframer-docs-public/fraud-detection")

In [3]:
!pip install pandas pydataframer tenacity requests

import io
import os
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import requests
from tenacity import retry, retry_if_result, stop_never, wait_fixed

import dataframer
from dataframer import Dataframer

df_client = Dataframer(api_key=os.environ["DATAFRAMER_API_KEY"])

FILES_DIR = Path("files")

print(f"Dataframer SDK version: {dataframer.__version__}")

Dataframer SDK version: 0.7.0


## Load the Seed Dataset

The seed file contains 10 transactions from a single cardholder recorded on a single day. Reading through the rows tells the geo-jump fraud story clearly:

- The day starts normally — a grocery purchase in New York
- Less than an hour later, a transaction appears in **Los Angeles** — physically impossible if the previous one was a POS swipe in New York
- The city hops continue: New York → Los Angeles → Chicago → San Francisco → Miami, all within ~4 hours
- Each city transition is flagged: the elapsed time between transactions is far too short for any mode of travel between those cities

DataFramer will learn from the seed's structure and generate hundreds of realistic variations across different city pairs, time gaps, merchant categories, and fraud severities.

In [ ]:
csv_path = FILES_DIR / "transaction_seed_geo_jump.csv"
seed_df = pd.read_csv(csv_path)

print(f"Rows: {len(seed_df)}  |  Columns: {list(seed_df.columns)}")
print()
print("Geographic jump sequence:")
print(seed_df[["transaction_id", "event_timestamp", "city", "state", "amount"]].to_string(index=False))
print()
seed_df.head(5)

Rows: 10  |  Columns: ['transaction_id', 'customer_name', 'event_timestamp', 'amount', 'currency', 'merchant_category_code', 'city', 'state', 'country', 'payment_channel', 'auth_method', 'device_trust_level', 'txn_velocity_24h', 'days_since_last_txn', 'account_age_months', 'txns_today', 'declines_30d']

Geographic jump sequence:
transaction_id      event_timestamp          city      state  amount
      TXN-N-01 2026-03-12T08:45:00Z      new_york   new_york   38.00
      TXN-N-02 2026-03-12T09:05:30Z      new_york   new_york   62.00
      TXN-N-03 2026-03-12T09:58:12Z   los_angeles california  110.00
      TXN-N-04 2026-03-12T10:06:40Z   los_angeles california   27.00
      TXN-N-05 2026-03-12T11:22:18Z       chicago   illinois   95.00
      TXN-N-06 2026-03-12T11:29:44Z       chicago   illinois   19.50
      TXN-N-07 2026-03-12T12:03:07Z san_francisco california  140.00
      TXN-N-08 2026-03-12T12:15:55Z san_francisco california   22.00
      TXN-N-09 2026-03-12T12:48:31Z         miam

,transaction_id,customer_name,event_timestamp,amount,currency,merchant_category_code,city,state,country,payment_channel,auth_method,device_trust_level,txn_velocity_24h,days_since_last_txn,account_age_months,txns_today,declines_30d
0,TXN-N-01,John Doe,2026-03-12T08:45:00Z,38.0,USD,5411,new_york,new_york,us,POS,PIN,high,1,0,25,1,0
1,TXN-N-02,John Doe,2026-03-12T09:05:30Z,62.0,USD,5812,new_york,new_york,us,ECOM,OTP,high,2,0,25,2,0
2,TXN-N-03,John Doe,2026-03-12T09:58:12Z,110.0,USD,5732,los_angeles,california,us,ECOM,OTP,medium,3,0,25,3,0
3,TXN-N-04,John Doe,2026-03-12T10:06:40Z,27.0,USD,5411,los_angeles,california,us,POS,NONE,medium,4,0,25,4,0
4,TXN-N-05,John Doe,2026-03-12T11:22:18Z,95.0,USD,5814,chicago,illinois,us,ECOM,OTP,medium,5,0,25,5,0


---

## Generating a Specification with the DataFramer SDK

We upload the seed CSV to DataFramer, which analyses its structure and produces a [Specification](https://docs.dataframer.ai/essentials/specifications): a structured description of the data properties, value distributions, and shared requirements. That spec is the engine for generating 100 realistic geo-jump fraud transaction sequences.

### Step 1: Upload the CSV as a Seed Dataset

The CSV is packed into a ZIP in memory and uploaded to DataFramer.

In [3]:
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, arcname=csv_path.name)
    print(f"  Added: {csv_path.name}")
zip_buffer.seek(0)

dataset = df_client.dataframer.seed_datasets.create_from_zip(
    name=f"fraud_geo_jump_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
    description="Seed transactions showing impossible geographic jump fraud patterns",
    zip_file=zip_buffer,
)

dataset_id = dataset.id
print(f"\nUpload complete")
print(f"Dataset ID: {dataset_id}")
print(f"Files: {dataset.file_count}")

  Added: transaction_seed_velocity_geo_jump.csv

Upload complete
Dataset ID: eb288aa9-aa26-4867-a519-72ab17f2df2f
Files: 1


### Step 2: Generate a Specification

DataFramer analyses the seed rows and produces a spec that captures:
- The column structure and data types
- The geographic jump pattern — city/state sequences where the elapsed time between consecutive transactions is too short for any mode of travel
- The `geo_jump_flag` property to label whether a transaction involves an impossible location transition
- The distribution of merchant categories, payment channels, and auth methods
- Amount patterns by merchant category

We pass generation objectives to ensure the spec explicitly captures the geo-jump-specific properties we want reproduced at scale.

In [ ]:
SPEC_ID = None  # Set to an existing spec ID to skip creation

if SPEC_ID is None:
    spec_name = f"fraud_geo_jump_spec_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    spec = df_client.dataframer.specs.create(
        dataset_id=dataset_id,
        name=spec_name,
        spec_generation_model_name="anthropic/claude-sonnet-4-6",
        generation_objectives=(
            "Do include 'geo_jump_flag' as a data property with values: normal, geo_jump — "
            "indicating whether the transaction involves a city transition that is physically "
            "impossible within the elapsed time since the previous transaction. "
            "Split 'event_timestamp' into discrete integer fields that must be called: year, month, day, hour, "
            "minute, and second. These must form a valid calendar date and time. "
            "The fields 'txn_velocity_24h', 'days_since_last_txn', and 'txns_today' must NOT "
            "be generated as static values; they are derived calculated fields based on the "
            "timestamp and transaction ordering. "
            "The 'country' is always 'us'. The 'currency' is always 'USD'."
        ),
        extrapolate_values=True,
        generate_distributions=True,
    )
    SPEC_ID = spec.id
    print(f"Started spec analysis: {spec_name}")

    def spec_not_ready(result):
        return result.status not in ("SUCCEEDED", "FAILED")

    @retry(wait=wait_fixed(5), retry=retry_if_result(spec_not_ready), stop=stop_never)
    def poll_spec_status(spec_id):
        return df_client.dataframer.specs.retrieve(spec_id=spec_id)

    print("Polling for spec status...")
    spec_result = poll_spec_status(SPEC_ID)

    if spec_result.status == "FAILED":
        raise RuntimeError("Spec analysis failed")

    print(f"\nSpec analysis completed successfully!")

spec_id = SPEC_ID
print(f"Spec ID: {spec_id}")

Spec ID: 33e3328a-8ffb-4fb7-aa4b-971fcb4d8bcd


### Step 3: Review the Analysed Data Properties & Shared Properties

DataFramer distinguishes between:
- **Data property variations**: per-column properties with observed value distributions (e.g. geo jump flag, merchant category, payment channel). Expect to see `Geo Jump Flag` as an explicit property capturing impossible location transitions.
- **Shared properties / requirements**: cross-row rules that every generated sample must satisfy (e.g. city transitions flagged as `geo_jump` must have an elapsed time shorter than the minimum travel time between the two cities by any mode of transport).

In [32]:
import textwrap
import yaml

spec = df_client.dataframer.specs.retrieve(spec_id=spec_id)
config = yaml.safe_load(spec.content_yaml)
spec_data = config.get("spec", config)

print()
print("=" * 60)
print("SHARED REQUIREMENTS / PROPERTIES")
print("=" * 60)
shared = spec_data.get("requirements") or spec_data.get("shared_requirements") or "(none found)"
print("\n".join(textwrap.fill(line, width=80) for line in shared.splitlines()))

print("=" * 60)
print("DATA PROPERTY VARIATIONS")
print("=" * 60)
for prop in spec_data.get("data_property_variations", []):
    values = prop.get("property_values", [])
    dists = prop.get("base_distributions", {})
    print(f"\n  {prop['property_name']}  ({len(values)} values)")
    for v in values:
        weight = dists.get(v, "–")
        print(f"    {v!r:45s}  weight: {weight}")

print()
print("Full spec YAML:")
print(spec.content_yaml)


SHARED REQUIREMENTS / PROPERTIES
Each data point represents a single financial transaction record for customer
'John Doe' in the US with currency USD. Each record contains the following
fields: transaction_id (string, format TXN-N-XX), customer_name (always 'John
Doe'), event_timestamp split into discrete integer fields: year (always 2026),
month (always 3), day (always 12), hour (integer), minute (integer), second
(integer); amount (decimal number in USD), currency (always 'USD'),
merchant_category_code (4-digit MCC code), city (lowercase string), state
(lowercase string), country (always 'us'), payment_channel, auth_method,
device_trust_level, txn_velocity_24h (derived: count of transactions in prior
24h window including this one), days_since_last_txn (derived: 0 if same day as
previous transaction, else days elapsed), txns_today (derived: count of
transactions on the same calendar day up to and including this one),
declines_30d (integer count of declines in past 30 days), and geo_j

### Step 4: Adjust the Geo Jump Flag Distribution

The 10-row seed is a single session of uniformly geo-jumping transactions. To build a training-ready dataset, we want a mix that reflects realistic fraud: a meaningful minority of transactions involve impossible jumps, while the majority are geographically plausible.

We override the `geo_jump_flag` base distribution for all city/states:

| Value | Weight | Represents |
|---|---|---|
| `normal` | 60 | Geographically plausible transactions i.e. same city or nearby cities within a realistic travel window |
| `geo_jump` | 40 | Impossible location transitions i.e. city pairs that cannot be reached within the elapsed time by any mode of transport |

**Conditional Distribution on State**

Geographic jump activity is not evenly distributed across the US. High-traffic metropolitan states attract disproportionate fraud because they have the most legitimate transaction noise to hide in. We add a conditional distribution so that geo-jump rates vary by state:

| State | `geo_jump` weight | `normal` weight | Rationale |
|---|---|---|---|
| `new_york` | 90 | 10 | Dense metropolitan area, high card-not-present volume |
| `california` | 90 | 10 | Largest state economy, major fraud hub |
| `florida` | 90 | 10 | High tourism traffic creates cover for impossible travel |

In [35]:
import copy

GEO_PROP = "geo_jump_flag"

GEO_DISTRIBUTIONS = {
    "normal": 60,
    "geo_jump": 40,
}

GEO_STATE_CONDITIONAL_DISTRIBUTIONS = {
    "state": {
        "new_york":   {"geo_jump": 90, "normal": 10},
        "california": {"geo_jump": 90, "normal": 10},
        "florida":    {"geo_jump": 90, "normal": 10},
    }
}

updated_spec_data = copy.deepcopy(spec_data)

for prop in updated_spec_data.get("data_property_variations", []):
    if prop["property_name"] == GEO_PROP:
        prop["base_distributions"] = {
            v: GEO_DISTRIBUTIONS.get(v, 0)
            for v in prop["property_values"]
        }
        prop["conditional_distributions"] = GEO_STATE_CONDITIONAL_DISTRIBUTIONS
        print(f"Updated '{GEO_PROP}':")
        for k, w in prop["base_distributions"].items():
            print(f"  {k!r:20s}  {w}")
        print(f"\nConditional distributions on state:")
        for state, dists in GEO_STATE_CONDITIONAL_DISTRIBUTIONS["state"].items():
            print(f"  state={state!r}: {dists}")
        break
else:
    print(f"WARNING: '{GEO_PROP}' not found in spec — adding it")
    updated_spec_data.setdefault("data_property_variations", []).append({
        "property_name": GEO_PROP,
        "property_values": list(GEO_DISTRIBUTIONS.keys()),
        "base_distributions": GEO_DISTRIBUTIONS,
        "conditional_distributions": GEO_STATE_CONDITIONAL_DISTRIBUTIONS,
    })

updated_yaml = yaml.dump({"spec": updated_spec_data}, allow_unicode=True, sort_keys=False)

updated_spec = df_client.dataframer.specs.update(spec_id=spec_id, content_yaml=updated_yaml)

print(f"\nUpdated spec saved")
print(f"Spec ID: {updated_spec.id}")
print(f"Status:  {updated_spec.status}")

Updated 'geo_jump_flag':
  'normal'              60
  'geo_jump'            40

Conditional distributions on state:
  state='new_york': {'geo_jump': 90, 'normal': 10}
  state='california': {'geo_jump': 90, 'normal': 10}
  state='florida': {'geo_jump': 90, 'normal': 10}

Updated spec saved
Spec ID: 33e3328a-8ffb-4fb7-aa4b-971fcb4d8bcd
Status:  SUCCEEDED


### Step 5: Generate 100 Fraud Transaction Samples

Each sample is a complete, coherent row — a single transaction — generated by Claude following the spec constraints:

- `geo_jump` transactions appear in cities that are impossible to reach within the elapsed time from the previous transaction
- `normal` transactions stay in the same city or move to nearby cities within a plausible travel window
- Merchant categories, payment channels, and auth methods vary naturally
- Amount patterns vary by merchant category

Revision passes enforce consistency and conformance, catching any samples where the city transition is borderline plausible or the `geo_jump_flag` is inconsistent with the elapsed time.

In [ ]:
RUN_ID = None  # Set to an existing run ID to skip creation

if RUN_ID is None:
    run = df_client.dataframer.runs.create(
        spec_id=updated_spec.id,
        number_of_samples=100,
        generation_model="anthropic/claude-haiku-4-5",
        revision_types=["consistency", "conformance"],
        max_revision_cycles=1,
        filtering_types=["conformance", "structural"],
        generation_thinking_budget=4000,
        skip_outline=True,
    )
    RUN_ID = run.id
    print(f"Run started")
    print(f"Run ID: {RUN_ID}")

    def run_not_ready(result):
        return result.status not in ("SUCCEEDED", "FAILED")

    @retry(wait=wait_fixed(10), retry=retry_if_result(run_not_ready), stop=stop_never)
    def poll_run_status(run_id):
        return df_client.dataframer.runs.retrieve(run_id=run_id)

    print("Polling for run status...")
    run_result = poll_run_status(RUN_ID)

    if run_result.status == "FAILED":
        raise RuntimeError("Run failed")

    print(f"\nRun completed successfully!")
    print(f"Samples completed: {run_result.samples_completed}")
    print(f"Samples failed:    {run_result.samples_failed}")

run_id = RUN_ID
print(f"Run ID:            {run_id}")

Run started
Run ID: dfbf76c0-2ce4-458f-906c-bad1293440a4
Polling for run status...

Run completed successfully!
Samples completed: 100
Samples failed:    0
Run ID:            dfbf76c0-2ce4-458f-906c-bad1293440a4


### Step 6: Download Generated Samples & Check Cost

In [16]:
base_url = str(df_client.base_url).rstrip("/")
cost_data = requests.get(
    f"{base_url}/api/dataframer/runs/{run_id}/cost/",
    headers={"Authorization": f"Bearer {os.environ['DATAFRAMER_API_KEY']}"},
).json()

total_cost = cost_data.get("total_cost_cents")
print(f"Total cost for 100 samples: ${total_cost / 100:.2f}")


def get_download_url(result):
    if isinstance(result, str):
        return result
    return result.download_url


def zip_not_ready(result):
    return get_download_url(result) is None


@retry(wait=wait_fixed(5), retry=retry_if_result(zip_not_ready), stop=stop_never)
def poll_download_all(run_id):
    return df_client.dataframer.runs.files.download_all(run_id=run_id)


print("\nGenerating download ZIP...")
dl = poll_download_all(run_id)
download_url = get_download_url(dl)
print("ZIP ready")

output_dir = Path("output") / run_id
output_dir.mkdir(parents=True, exist_ok=True)

zip_data = requests.get(download_url).content
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall(output_dir)
    names = sorted(zf.namelist())

print(f"\nExtracted {len(names)} files to {output_dir}/")
for name in names:
    print(f"  {name}")

Total cost for 100 samples: $8.64

Generating download ZIP...
ZIP ready

Extracted 2 files to output/dfbf76c0-2ce4-458f-906c-bad1293440a4/
  generated_samples.csv
  top_level.metadata


---

## Explore the Generated Dataset

Let's load all generated CSVs and verify that the geo-jump pattern is present, correctly distributed, and that city sequences are coherent with the `geo_jump_flag`.

In [17]:
generated_dfs = []
for csv_file in sorted(output_dir.glob("*.csv")):
    generated_dfs.append(pd.read_csv(csv_file))

generated_df = pd.concat(generated_dfs, ignore_index=True)
generated_df.columns = generated_df.columns.str.lower()

print(f"Total generated samples: {len(generated_df)}")
print(f"Columns: {list(generated_df.columns)}")
print()
generated_df.head(5)

Total generated samples: 100
Columns: ['transaction_id', 'customer_name', 'event_timestamp', 'amount', 'currency', 'merchant_category_code', 'city', 'state', 'country', 'payment_channel', 'auth_method', 'device_trust_level', 'txn_velocity_24h', 'days_since_last_txn', 'account_age_months', 'txns_today', 'declines_30d']



,transaction_id,customer_name,event_timestamp,amount,currency,merchant_category_code,city,state,country,payment_channel,auth_method,device_trust_level,txn_velocity_24h,days_since_last_txn,account_age_months,txns_today,declines_30d
0,TXN-N-05,John Doe,2026-03-12T08:20:02Z,75.0,USD,5812,philadelphia,pennsylvania,us,ECOM,NONE,medium,5,0,25,5,0
1,TXN-N-14,John Doe,2026-03-12T10:15:31Z,180.0,USD,5411,denver,colorado,us,POS,PIN,medium,14,0,25,14,0
2,TXN-N-08,John Doe,2026-03-12T10:52:44Z,75.0,USD,5311,seattle,washington,us,POS,PIN,low,8,0,25,8,0
3,TXN-N-15,John Doe,2026-03-12T16:56:25Z,45.0,USD,5945,philadelphia,pennsylvania,us,ECOM,OTP,medium,11,0,25,11,0
4,TXN-N-14,John Doe,2026-03-12T11:03:03Z,27.0,USD,5411,los_angeles,california,us,MOBILE,BIOMETRIC,high,14,0,25,14,1


In [26]:
import math

CITY_COORDS = {
    "new_york":      (40.7128,  -74.0060),
    "los_angeles":   (34.0522, -118.2437),
    "chicago":       (41.8781,  -87.6298),
    "san_francisco": (37.7749, -122.4194),
    "miami":         (25.7617,  -80.1918),
    "houston":       (29.7604,  -95.3698),
    "phoenix":       (33.4484, -112.0740),
    "philadelphia":  (39.9526,  -75.1652),
    "san_antonio":   (29.4241,  -98.4936),
    "dallas":        (32.7767,  -96.7970),
    "seattle":       (47.6062, -122.3321),
    "boston":        (42.3601,  -71.0589),
    "denver":        (39.7392, -104.9903),
    "atlanta":       (33.7490,  -84.3880),
    "las_vegas":     (36.1699, -115.1398),
    "portland":      (45.5051, -122.6750),
    "minneapolis":   (44.9778,  -93.2650),
    "detroit":       (42.3314,  -83.0458),
    "nashville":     (36.1627,  -86.7816),
    "orlando":       (28.5383,  -81.3792),
}

# Faster than any commercial aircraft — physically impossible overland or by air
GEO_JUMP_SPEED_KMH = 500

def haversine_km(city1, city2):
    lat1, lon1 = CITY_COORDS[city1]
    lat2, lon2 = CITY_COORDS[city2]
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2) ** 2)
    return R * 2 * math.asin(math.sqrt(a))

generated_df["timestamp"] = pd.to_datetime(generated_df["event_timestamp"], utc=True)

generated_df = generated_df.sort_values("timestamp").reset_index(drop=True)

# Consecutive-transition features
generated_df["prev_city"]      = generated_df["city"].shift(1)
generated_df["prev_timestamp"] = generated_df["timestamp"].shift(1)
generated_df["time_delta_h"]   = (
    (generated_df["timestamp"] - generated_df["prev_timestamp"]).dt.total_seconds() / 3600
)
generated_df["distance_km"] = generated_df.apply(
    lambda r: haversine_km(r["prev_city"], r["city"])
    if (pd.notna(r["prev_city"])
        and r["city"] != r["prev_city"]
        and r["prev_city"] in CITY_COORDS
        and r["city"] in CITY_COORDS)
    else 0.0,
    axis=1,
)
generated_df["implied_speed_kmh"] = generated_df.apply(
    lambda r: r["distance_km"] / r["time_delta_h"] if r["time_delta_h"] < 2 else float("inf"),
    axis=1,
)
generated_df["geo_jump_detected"] = generated_df["implied_speed_kmh"] > GEO_JUMP_SPEED_KMH

unknown_cities = set(generated_df["city"].unique()) - CITY_COORDS.keys()
if unknown_cities:
    print(f"WARNING: cities not in CITY_COORDS (treated as no-jump): {sorted(unknown_cities)}")
    print()

print("Derived geo-jump distribution:")
print(generated_df["geo_jump_detected"].value_counts().rename({True: "geo_jump", False: "normal"}).to_string())
print()

print("Payment channel distribution:")
print(generated_df["payment_channel"].value_counts().to_string())
print()

print("Auth method distribution:")
print(generated_df["auth_method"].value_counts().to_string())
print()

print("Top cities:")
print(generated_df["city"].value_counts().head(10).to_string())


Derived geo-jump distribution:
geo_jump_detected
geo_jump    85
normal      15

Payment channel distribution:
payment_channel
ECOM           40
POS            29
CONTACTLESS    14
MOBILE         10
ATM             7

Auth method distribution:
auth_method
PIN          33
OTP          26
NONE         16
BIOMETRIC    15
PASSWORD      8
CVV           2

Top cities:
city
san_francisco    14
los_angeles      12
chicago          12
new_york         11
seattle           7
dallas            6
philadelphia      5
denver            5
houston           5
boston            4


In [42]:
import argparse
import csv
import re
from datetime import datetime, timedelta


def parse_timestamp_with_invalid_day_fix(timestamp_str):
    """Parse timestamp, fixing invalid days like June 31 by clamping to last valid day."""
    ts = timestamp_str.replace('Z', '+00:00')
    try:
        return datetime.fromisoformat(ts)
    except ValueError as e:
        match = re.search(r'day (\d+) must be in range 1\.\.(\d+) for month (\d+) in year (\d+)', str(e))
        if match:
            invalid_day, max_day, month, year = map(int, match.groups())
            fixed_ts = re.sub(
                r'(\d{4})-(\d{2})-(\d{2})',
                lambda m: f"{m.group(1)}-{m.group(2)}-{max_day:02d}",
                ts,
                count=1
            )
            return datetime.fromisoformat(fixed_ts)
        raise


def fix_transaction_counters(input_file, output_file):
    transactions = []
    with open(input_file, 'r', newline='') as infile:
        reader = csv.DictReader(infile)
        for row in reader:
            for key in ['amount', 'txn_velocity_24h', 'days_since_last_txn', 'account_age_months', 'txns_today', 'declines_30d']:
                try:
                    row[key] = float(row[key]) if '.' in row[key] else int(row[key])
                except (ValueError, TypeError):
                    row[key] = None
            row['event_timestamp_dt'] = parse_timestamp_with_invalid_day_fix(row['event_timestamp'])
            transactions.append(row)

    transactions.sort(key=lambda x: (x['customer_name'], x['event_timestamp_dt']))

    customer_history = {}
    customer_first_txn = {}

    for i in range(len(transactions)):
        current_txn = transactions[i]
        customer_name = current_txn['customer_name']
        current_timestamp = current_txn['event_timestamp_dt']

        if customer_name not in customer_history:
            customer_history[customer_name] = []
            baseline_age = current_txn['account_age_months'] if current_txn['account_age_months'] is not None else 0
            customer_first_txn[customer_name] = {
                'timestamp': current_timestamp,
                'baseline_account_age': baseline_age
            }

        history = customer_history[customer_name]

        first_txn_data = customer_first_txn[customer_name]
        first_timestamp = first_txn_data['timestamp']
        baseline_age = first_txn_data['baseline_account_age']
        months_diff = (current_timestamp.year - first_timestamp.year) * 12 + (current_timestamp.month - first_timestamp.month)
        if current_timestamp.day < first_timestamp.day:
            months_diff -= 1
        current_txn['account_age_months'] = baseline_age + max(0, months_diff)

        if history:
            last_txn_timestamp = history[-1]
            current_txn['days_since_last_txn'] = (current_timestamp - last_txn_timestamp).days
        else:
            current_txn['days_since_last_txn'] = 0

        txns_today_count = 0
        for txn_time in reversed(history):
            if txn_time.date() == current_timestamp.date():
                txns_today_count += 1
            else:
                break
        current_txn['txns_today'] = txns_today_count + 1

        one_day_ago = current_timestamp - timedelta(days=1)
        txn_velocity_24h_count = 0
        for txn_time in reversed(history):
            if txn_time > one_day_ago:
                txn_velocity_24h_count += 1
            else:
                break
        current_txn['txn_velocity_24h'] = txn_velocity_24h_count + 1

        history.append(current_timestamp)

    for txn in transactions:
        del txn['event_timestamp_dt']
        if txn['amount'] is not None:
            txn['amount'] = f"{float(txn['amount']):.2f}"

    transactions.sort(key=lambda x: x['event_timestamp'])

    pad_width = max(2, len(str(len(transactions))))
    for idx, txn in enumerate(transactions, start=1):
        txn['transaction_id'] = f"TXN-{idx:0{pad_width}d}"

    with open(output_file, 'w', newline='') as outfile:
        if transactions:
            writer = csv.DictWriter(outfile, fieldnames=transactions[0].keys())
            writer.writeheader()
            writer.writerows(transactions)


# Apply to all generated CSVs
fixed_dfs = []
for csv_file in sorted(output_dir.glob("*.csv")):
    fixed_path = csv_file.with_stem(csv_file.stem + "_fixed")
    fix_transaction_counters(str(csv_file), str(fixed_path))
    fixed_dfs.append(pd.read_csv(fixed_path))

fixed_df = pd.concat(fixed_dfs, ignore_index=True)
print(f"Fixed counter columns written alongside each generated CSV")
print()
fixed_df.head(5)

Fixed counter columns written alongside each generated CSV



,transaction_id,customer_name,event_timestamp,amount,currency,merchant_category_code,city,state,country,payment_channel,auth_method,device_trust_level,txn_velocity_24h,days_since_last_txn,account_age_months,txns_today,declines_30d
0,TXN-001,John Doe,2026-03-12T06:25:41Z,52.0,USD,5999,san_francisco,california,us,ECOM,PASSWORD,high,1,0,25,1,1
1,TXN-002,John Doe,2026-03-12T07:00:42Z,22.0,USD,5812,dallas,texas,us,ECOM,OTP,high,2,0,25,2,2
2,TXN-003,John Doe,2026-03-12T07:02:01Z,85.0,USD,5499,chicago,illinois,us,POS,NONE,medium,3,0,25,3,0
3,TXN-004,John Doe,2026-03-12T07:02:24Z,75.0,USD,5499,new_york,new_york,us,MOBILE,NONE,medium,4,0,25,4,0
4,TXN-005,John Doe,2026-03-12T07:18:33Z,140.0,USD,5732,new_york,new_york,us,ECOM,OTP,high,5,0,25,5,3


In [43]:
print("Amount and velocity stats by geo_jump_detected:")
print(
    generated_df.groupby("geo_jump_detected")[["txn_velocity_24h", "txns_today", "amount"]]
    .agg(["mean", "max"])
    .round(2)
    .rename(index={True: "geo_jump", False: "normal"})
    .to_string()
)
print()

print("City distribution by geo_jump_detected:")
print(
    generated_df.groupby("geo_jump_detected")["city"]
    .value_counts()
    .groupby(level=0)
    .head(5)
    .rename(index={True: "geo_jump", False: "normal"}, level=0)
    .to_string()
)

Amount and velocity stats by geo_jump_detected:
                  txn_velocity_24h     txns_today      amount       
                              mean max       mean max    mean    max
geo_jump_detected                                                   
normal                       11.83  20      12.33  20  112.86  400.0
geo_jump                      9.67  20       9.65  20   90.65  400.0

City distribution by geo_jump_detected:
geo_jump_detected  city         
normal             san_diego         4
                   new_york          3
                   los_angeles       3
                   austin            2
                   boston            2
geo_jump           san_francisco    12
                   new_york         11
                   los_angeles      10
                   chicago          10
                   houston           7


In [ ]:
print("Sample geo-jump transitions (derived):")
geo_jumps = generated_df[generated_df["geo_jump_detected"]].copy()
print(
    geo_jumps[["transaction_id", "timestamp", "prev_city", "city", "distance_km", "time_delta_h", "implied_speed_kmh", "amount"]]
    .head(10)
    .round(1)
    .to_string(index=False)
)

---

## What's Next?

With 100 labelled fraud transaction rows in hand, natural next steps are:

- **Train a geo-anomaly classifier** — use `geo_jump_flag` as the label and location/time features (city, state, elapsed time since last transaction, distance between consecutive cities) as inputs. Gradient boosting models (XGBoost, LightGBM) handle tabular fraud data well.
- **Tune the geo-jump ratio** — adjust the `geo_jump_flag` distribution in Step 4 to match your production fraud prevalence and regenerate. A 5/95 jump-to-normal ratio more closely mirrors real-world imbalance.
- **Add more fraud patterns** — use DataFramer to generate separate datasets for velocity bursts and midnight crossing exploits, then combine them into a multi-pattern training set.
- **Stress-test rule engines** — feed the geo-jump samples into an existing location rule to find threshold gaps (does your rule trigger on a 1-hour cross-country jump? what about a 3-hour gap between distant cities?).
- **Publish to Hugging Face** — the dataset is clean, labelled, and ready to share.

---

### Suggested HuggingFace Dataset Name

```
dataframer/fraud-transactions-geo-jump
```

**Description tag line**: *Synthetic credit card fraud transactions exhibiting impossible geographic jumps — generated with DataFramer from a 10-row seed.*

**Tags**: `fraud-detection`, `synthetic`, `tabular`, `finance`, `classification`, `transaction-data`